# Ablation Experiments for Tiny LLM

Systematic hyperparameter search using a phased approach:

0. **Phase 0: Micro Screening** — cheap ~7M-param experiments to eliminate bad architectural choices
0. **Phase 0.5: Depth vs Width** — depth/width trade-offs at 12M and 19M scales
1. **Phase 1: Architecture Shape** — depth, width, FFN ratio at ~37-40M params
1.5 **Phase 1.5: Shape C Refinement** — local depth/FFN search around the best Shape C template
2. **Phase 2: Learning Rate** — sweep using the Phase 1.5 winner (fallback: Phase 1 winner)
3. **Phase 3: Coupled Ablations** — dropout, QK-norm, activation (GELU vs SwiGLU)

Phases 0 and 0.5 use **single seed** for fast screening.
Phases 1, 1.5, 2, and 3 use **multiple random seeds** (default: 3) for statistical robustness.

This notebook launches each training run in a fresh subprocess via `main.py --stage train`,
which avoids CUDA-memory accumulation across many experiments while keeping settings identical.

### Ablation Step Budget

| Phase | Steps | Eval Every | Batch Config | Rationale |
|-------|------:|----------:|:------------|-----------|
| Phase 0 (micro) | 1,000 | 100 | 16×2 (32K tok/step) | Cheap screening, fits any device |
| Phase 0.5 (medium) | 2,000 | 200 | 16×2 (32K tok/step) | Depth/width at medium scale |
| Phase 1 (architecture) | 1,000 | 100 | device-aware | Shape differences separate early |
| Phase 1.5 (refinement) | 1,000 | 100 | device-aware | Local search around Shape C without rerunning baseline |
| Phase 2 (learning rate) | 1,500 | 150 | device-aware | Need past warmup + main descent |
| Phase 3 (regularization) | 2,000 | 200 | device-aware | Train/val gap takes longer to show |

Architecture rankings typically stabilise within 5-10% of full compute
(Hoffmann et al., 2022), so these budgets are sufficient for reliable ranking.



In [ ]:
# ── Environment setup (Colab-safe) ───────────────────────────────────
import os, sys, math, getpass, zipfile, shutil, urllib.request, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")

    # Clone repo and install dependencies
    if not os.path.exists("40m-tiny-llm"):
        token = os.environ.get("GITHUB_TOKEN") or getpass.getpass("GitHub Token: ")
        subprocess.run(
            ["git", "clone", f"https://{token}@github.com/marcbloech/40m-tiny-llm.git"],
            check=True,
        )
    os.chdir("40m-tiny-llm")
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[train]", "-q"], check=True)

    data_dir = Path("data")
    gdrive_dir = Path("/content/drive/MyDrive/tiny_llm")
    gdrive_zip = gdrive_dir / "data.zip"

    # Ablation target (full) and Colab-safe target
    TOKENS_PER_STEP = 64 * 4 * 1024
    LONGEST_PHASE_STEPS = 2000
    FULL_TARGET_TRAIN_TOKENS = int(TOKENS_PER_STEP * LONGEST_PHASE_STEPS * 1.20)  # 629,145,600
    COLAB_SAFE_TARGET_TRAIN_TOKENS = 300_000_000
    TARGET_TRAIN_TOKENS = min(FULL_TARGET_TRAIN_TOKENS, COLAB_SAFE_TARGET_TRAIN_TOKENS)

    # Conservative docs->tokens estimate
    EST_TOKENS_PER_DOC = 700
    VAL_FRACTION = 0.05
    MAX_SAMPLES_COLAB = 400_000

    def ensure_eval_set() -> None:
        eval_root = data_dir / "eval"
        test_dir = eval_root / "test"
        if test_dir.exists():
            return

        zip_url = "https://drive.switch.ch/index.php/s/6TLGQFEIkAPJ72K/download?path=%2F&files=eval.zip"
        zip_path = Path("/tmp/eval.zip")

        data_dir.mkdir(parents=True, exist_ok=True)
        if eval_root.exists():
            shutil.rmtree(eval_root)

        print("Eval set not found, downloading eval-test ZIP...")
        urllib.request.urlretrieve(zip_url, zip_path)

        print("Extracting eval-test ZIP...")
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(data_dir)
        zip_path.unlink(missing_ok=True)

        nested = eval_root / "eval"
        if nested.exists() and nested.is_dir():
            for p in nested.iterdir():
                shutil.move(str(p), str(eval_root / p.name))
            nested.rmdir()

        if not test_dir.exists():
            raise FileNotFoundError(f"Expected eval test set at {test_dir}, but it was not found after extraction.")

    def zip_and_copy_tokenized_data() -> None:
        local_zip = Path("/tmp/data.zip")
        if local_zip.exists():
            local_zip.unlink()

        include_roots = [data_dir / "tokenized", data_dir / "eval"]

        print("Creating data.zip (tokenized + eval set only)...")
        with zipfile.ZipFile(local_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
            for root in include_roots:
                if not root.exists():
                    raise FileNotFoundError(f"Missing path for ZIP: {root}")
                for p in root.rglob("*"):
                    if p.is_file():
                        zf.write(p, arcname=str(p))

        gdrive_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(local_zip, gdrive_zip)
        print(f"Copied data.zip to Google Drive: {gdrive_zip} ({local_zip.stat().st_size / 1e9:.2f} GB)")

    def run_data_stage(num_samples: int, force: bool = False) -> int:
        cmd = [
            "uv", "run", "python", "main.py", "--stage", "data",
            "--num-samples", str(num_samples),
            "--seed", "42",
            "--eval-test-dir", str(data_dir / "eval" / "test"),
        ]
        if force:
            cmd.append("--force")
        subprocess.run(cmd, check=True)
        train_bin = data_dir / "tokenized" / "train.bin"
        if not train_bin.exists():
            raise FileNotFoundError(f"{train_bin} was not generated.")
        return train_bin.stat().st_size // 2  # uint16 => 2 bytes/token

    # Choose data setup mode
    mode = os.environ.get("DATA_MODE", "").strip().lower()
    if mode not in {"retokenize", "drive_zip"}:
        choice = input(
            "Choose data source: [1] re-download + tokenize, [2] data.zip from Google Drive (default: 2): "
        ).strip()
        mode = "retokenize" if choice == "1" else "drive_zip"

    if mode == "retokenize":
        if data_dir.exists():
            shutil.rmtree(data_dir)
        data_dir.mkdir(parents=True, exist_ok=True)

        ensure_eval_set()

        estimated_samples = math.ceil(TARGET_TRAIN_TOKENS / ((1.0 - VAL_FRACTION) * EST_TOKENS_PER_DOC))
        start_samples = min(max(100_000, estimated_samples), MAX_SAMPLES_COLAB)

        # Fallback ladder if Colab kills the process (exit code 137)
        candidates = [start_samples, 300_000, 240_000, 180_000, 140_000]
        candidates = [x for i, x in enumerate(candidates) if x <= MAX_SAMPLES_COLAB and x >= 100_000 and x not in candidates[:i]]

        print(f"Full ablation target: {FULL_TARGET_TRAIN_TOKENS:,} train tokens")
        print(f"Colab-safe target:    {TARGET_TRAIN_TOKENS:,} train tokens")
        print(f"Starting with --num-samples={start_samples:,} (cap={MAX_SAMPLES_COLAB:,})")

        train_tokens = None
        for i, n in enumerate(candidates, start=1):
            print(f"[Try {i}/{len(candidates)}] Running data stage with --num-samples={n:,} ...")
            try:
                train_tokens = run_data_stage(n, force=(i > 1))
                print(f"Success: generated {train_tokens:,} train tokens")
                break
            except subprocess.CalledProcessError as e:
                if e.returncode == 137:
                    print("Process was killed (OOM, exit 137). Trying a smaller sample size...")
                    continue
                raise

        if train_tokens is None:
            raise RuntimeError("All attempts failed due to OOM. Use a smaller runtime load or use drive_zip mode.")

        if train_tokens < TARGET_TRAIN_TOKENS:
            print(
                f"Warning: generated {train_tokens:,} train tokens, below target {TARGET_TRAIN_TOKENS:,}. "
                "This is still usable, but ablation stability may be lower."
            )

        zip_and_copy_tokenized_data()

    else:
        local_zip = Path("/tmp/data.zip")

        if not gdrive_zip.exists():
            raise FileNotFoundError(
                f"{gdrive_zip} not found. Please verify that data.zip exists in 'MyDrive/tiny_llm'."
            )

        if data_dir.exists():
            shutil.rmtree(data_dir)

        shutil.copy2(gdrive_zip, local_zip)
        print(f"Copying and extracting {gdrive_zip} ...")

        with zipfile.ZipFile(local_zip, "r") as zf:
            names = zf.namelist()
            contains_data_root = any(n.startswith("data/") for n in names)
            if contains_data_root:
                zf.extractall(".")
            else:
                data_dir.mkdir(parents=True, exist_ok=True)
                zf.extractall(data_dir)

        local_zip.unlink(missing_ok=True)
        print("Pre-tokenized data loaded from Google Drive.")

        ensure_eval_set()

    # Final checks
    eval_root = data_dir / "eval"
    test_dir = eval_root / "test"
    train_bin = data_dir / "tokenized" / "train.bin"

    if not test_dir.exists():
        raise FileNotFoundError(f"Expected eval test set at {test_dir}, but it was not found.")
    if not train_bin.exists():
        raise FileNotFoundError(f"Expected tokenized train set at {train_bin}, but it was not found.")

    print(f"Eval set destination: {eval_root.resolve()}")
    print(f"Train tokens: {train_bin.stat().st_size // 2:,}")
    print(f"Data mode: {mode}")

# Ensure src/ is importable
_cwd = Path(os.getcwd()).resolve()
REPO_ROOT = next(
    (p for p in (_cwd, *_cwd.parents) if (p / "pyproject.toml").exists() and (p / "src").exists()),
    _cwd,
)
os.chdir(REPO_ROOT)
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Repo root: {REPO_ROOT}")
print(f"Working dir: {os.getcwd()}")

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────
import json
import math
import time
from pathlib import Path

import torch
import pandas as pd
import matplotlib.pyplot as plt

from tiny_llm.utils.config import load_config, merge_cli_overrides
from tiny_llm.utils.device import resolve_device
from tiny_llm.utils.seed import set_seed
from tiny_llm.utils.logging import setup_logging
from tiny_llm.model.transformer import GPTModel
from tiny_llm.data.dataset import MemmapDataset
from tiny_llm.training.trainer import Trainer

setup_logging()

device = resolve_device("auto")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── Configuration ────────────────────────────────────────────────────

CONFIG_PATH = "configs/final.toml"   # base config (SwiGLU, RoPE, weight_tying)
DEVICE = "auto"                       # "auto", "cuda", "mps", "cpu"
SEEDS = [42, 123, 7]                  # multiple seeds for statistical robustness
                                      # Use SEEDS = [42] for fast single-seed screening
RUN_GROUP = globals().get("RUN_GROUP", f"run_{time.strftime('%Y%m%d')}")
RUNS_DIR = Path("runs") / RUN_GROUP
RUNS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Run output directory: {RUNS_DIR.resolve()}")

# ── Per-phase step budgets ────────────────────────────────────────────
# Phase 0: Micro screening — cheap, fast elimination
PHASE_0_STEPS = 1000
PHASE_0_EVAL  = 100

# Phase 0.5: Depth/width at medium scale
PHASE_05_STEPS = 2000
PHASE_05_EVAL  = 200

# Architecture: rankings stabilise by ~500-1000 steps
PHASE_1_STEPS = 1000
PHASE_1_EVAL  = 100

# Phase 1.5: local refinement around Shape C at the same budget
PHASE_15_STEPS = 1000
PHASE_15_EVAL  = 100

# LR: need ~1000-1500 steps past warmup to see curvature
PHASE_2_STEPS = 1500
PHASE_2_EVAL  = 150

# Regularization: train/val divergence needs more steps
PHASE_3_STEPS = 2000
PHASE_3_EVAL  = 200

# ── Device-aware batch sizing ─────────────────────────────────────────
# Tune train/eval batch sizes by hardware tier.
# Eval batch is kept conservative to avoid CUDA OOM during validation.

if device.type == "cuda":
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if vram_gb >= 70:
        # A100 80GB class
        BATCH_SIZE, GRAD_ACCUM, EVAL_BATCH_SIZE = 64, 4, 32
    elif vram_gb >= 35:
        # L40/A100 40GB class
        BATCH_SIZE, GRAD_ACCUM, EVAL_BATCH_SIZE = 32, 4, 16
    else:
        # 16-24GB class
        BATCH_SIZE, GRAD_ACCUM, EVAL_BATCH_SIZE = 16, 4, 8
elif device.type == "mps":
    BATCH_SIZE, GRAD_ACCUM, EVAL_BATCH_SIZE = 16, 4, 8
else:
    BATCH_SIZE, GRAD_ACCUM, EVAL_BATCH_SIZE = 8, 8, 4

print(
    f"Batch config: BATCH_SIZE={BATCH_SIZE}, GRAD_ACCUM={GRAD_ACCUM}, "
    f"EVAL_BATCH_SIZE={EVAL_BATCH_SIZE} "
    f"(~{BATCH_SIZE * GRAD_ACCUM * 1024 // 1024}K tokens/step)"
)

CHECKPOINT_INTERVAL = 50_000   # effectively "never" — only best.pt matters
SAMPLE_INTERVAL = 50_000       # disable sample generation for speed


In [ ]:
# ── Experiment runner ────────────────────────────────────────────────

import gc
import os
import re
import subprocess
import sys
import numpy as np

all_results: list[dict] = []
PHASE_PLOT_FILES: list[str] = []

_SEP_HEAVY = "━" * 60
_SEP_DOUBLE = "═" * 60


def _slugify(text: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", text.lower()).strip("_")


def _save_phase_plot(fig, title: str, phase_hint: str) -> Path:
    out_dir = RUNS_DIR / "visualizations" / "phase_plots"
    out_dir.mkdir(parents=True, exist_ok=True)
    file_name = f"{phase_hint}_{_slugify(title)}_val_ppl.png"
    plot_path = out_dir / file_name
    fig.savefig(plot_path, dpi=180, bbox_inches="tight")

    plot_str = str(plot_path)
    if plot_str not in PHASE_PLOT_FILES:
        PHASE_PLOT_FILES.append(plot_str)
    return plot_path


def _build_override_list(
    overrides: dict,
    max_steps: int,
    eval_interval: int,
    run_dir: Path,
) -> list[str]:
    override_list = [f"{k}={v}" for k, v in overrides.items()]
    override_list += [
        f"training.max_steps={max_steps}",
        f"training.eval_interval={eval_interval}",
        f"training.checkpoint_interval={CHECKPOINT_INTERVAL}",
        f"training.sample_interval={SAMPLE_INTERVAL}",
        f"training.checkpoint_dir={run_dir / 'checkpoints'}",
        "training.use_compile=false",  # keep parity with previous short-run setup
        "training.num_workers=0",      # notebook-safe: avoids multiprocessing worker teardown issues
    ]

    # Keep previous comparability: only inject global defaults when not explicitly overridden.
    if "training.batch_size" not in overrides:
        override_list.append(f"training.batch_size={BATCH_SIZE}")
    if "training.grad_accum_steps" not in overrides:
        override_list.append(f"training.grad_accum_steps={GRAD_ACCUM}")
    if "training.eval_batch_size" not in overrides:
        override_list.append(f"training.eval_batch_size={EVAL_BATCH_SIZE}")

    return override_list


def _estimate_n_params_from_overrides(override_list: list[str]) -> int:
    cfg = load_config(CONFIG_PATH)
    cfg = merge_cli_overrides(cfg, override_list)
    # Build on CPU only, no training state allocation.
    model = GPTModel(cfg.model.model_dump())
    n_params = sum(p.numel() for p in model.parameters())
    del model
    gc.collect()
    return n_params


def _run_train_subprocess(seed: int, override_list: list[str]) -> None:
    cmd = [
        sys.executable,
        "main.py",
        "--stage",
        "train",
        "--config",
        CONFIG_PATH,
        "--device",
        DEVICE,
        "--seed",
        str(seed),
        "--overrides",
        *override_list,
    ]

    env = os.environ.copy()
    src_path = str((Path.cwd() / "src").resolve())
    env["PYTHONPATH"] = src_path + (os.pathsep + env["PYTHONPATH"] if env.get("PYTHONPATH") else "")
    env.setdefault("MPLBACKEND", "Agg")
    env.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

    subprocess.run(cmd, check=True, env=env)


def _read_best_val_from_metrics(metrics_csv: Path) -> float:
    if not metrics_csv.exists():
        raise FileNotFoundError(f"Expected metrics file not found: {metrics_csv}")

    df = pd.read_csv(metrics_csv)
    if "val/loss" not in df.columns:
        raise KeyError(f"metrics.csv has no 'val/loss' column: {metrics_csv}")

    val_series = pd.to_numeric(df["val/loss"], errors="coerce").dropna()
    if val_series.empty:
        raise ValueError(f"No validation rows found in metrics.csv: {metrics_csv}")

    return float(val_series.min())


def run_ablation(
    run_name: str,
    overrides: dict,
    max_steps: int,
    eval_interval: int,
    seed: int = 42,
) -> dict:
    """Run one ablation experiment in a fresh subprocess.

    This keeps hyperparameters identical to prior runs while preventing GPU-memory
    accumulation across many experiments in a single notebook kernel.
    """
    run_dir = RUNS_DIR / run_name
    run_dir.mkdir(parents=True, exist_ok=True)

    override_list = _build_override_list(
        overrides=overrides,
        max_steps=max_steps,
        eval_interval=eval_interval,
        run_dir=run_dir,
    )

    n_params = _estimate_n_params_from_overrides(override_list)

    print(f"\n{_SEP_HEAVY}")
    print(f"  {run_name}  ({n_params / 1e6:.2f}M params, {max_steps} steps, seed={seed})")
    print(_SEP_HEAVY)

    t0 = time.time()
    _run_train_subprocess(seed=seed, override_list=override_list)
    runtime = time.time() - t0

    best_vl = _read_best_val_from_metrics(run_dir / "checkpoints" / "metrics.csv")

    result = {
        "run_name": run_name,
        "seed": seed,
        "n_params": n_params,
        "best_val_loss": round(best_vl, 4),
        "best_val_ppl": round(math.exp(best_vl), 2) if best_vl < 20 else float("inf"),
        "runtime_sec": round(runtime, 1),
        "max_steps": max_steps,
    }
    for k, v in overrides.items():
        result[k] = v

    (run_dir / f"{run_name}_metric.json").write_text(json.dumps(result, indent=2, default=str))
    all_results.append(result)

    print(
        f"\n  -> best_val_loss={result['best_val_loss']:.4f}  "
        f"val_ppl={result['best_val_ppl']:.2f}  "
        f"seed={seed}  ({runtime:.0f}s)"
    )

    # Notebook-process cleanup (subprocess already released train-time CUDA state).
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

    return result


def run_ablation_multi_seed(
    base_name: str,
    overrides: dict,
    max_steps: int,
    eval_interval: int,
    seeds: list[int] | None = None,
) -> list[dict]:
    """Run the same experiment across multiple seeds.

    Returns a list of per-seed result dicts.
    """
    seeds = seeds or SEEDS
    results = []
    for seed in seeds:
        run_name = f"{base_name}_s{seed}"
        result = run_ablation(
            run_name=run_name,
            overrides=overrides,
            max_steps=max_steps,
            eval_interval=eval_interval,
            seed=seed,
        )
        results.append(result)
    return results


def compare_phase(results: list[dict], title: str) -> pd.DataFrame:
    """Display a comparison table and bar chart for a set of ablation results.

    When multiple seeds are present per config, aggregates to mean +/- std and
    ranks by mean val_loss. The returned DataFrame contains one row per config
    with aggregated statistics.
    """
    df = pd.DataFrame(results)

    # Identify the config key: strip _s<seed> suffix to group by base experiment
    df["config_key"] = df["run_name"].str.replace(r"_s\d+$", "", regex=True)

    # Check if we actually have multiple seeds
    has_multi_seed = df.groupby("config_key")["seed"].nunique().max() > 1
    phase_hint = str(df["config_key"].iloc[0]).split("_")[0] if not df.empty else "phase"

    if has_multi_seed:
        # Aggregate across seeds
        extra_cols = sorted(
            c for c in df.columns
            if c.startswith("model.") or c.startswith("training.")
        )

        agg_rows = []
        for config_key, group in df.groupby("config_key"):
            row = {
                "config": config_key,
                "n_seeds": len(group),
                "n_params": group["n_params"].iloc[0],
                "mean_val_loss": group["best_val_loss"].mean(),
                "std_val_loss": group["best_val_loss"].std(),
                "mean_val_ppl": group["best_val_ppl"].mean(),
                "std_val_ppl": group["best_val_ppl"].std(),
                "mean_runtime": group["runtime_sec"].mean(),
            }
            # Carry over override columns from first row
            for c in extra_cols:
                if c in group.columns:
                    row[c] = group[c].iloc[0]
            agg_rows.append(row)

        agg_df = pd.DataFrame(agg_rows).sort_values("mean_val_loss")

        print(f"\n{_SEP_DOUBLE}")
        print(f"  {title}  ({agg_df['n_seeds'].iloc[0]} seeds per config)")
        print(_SEP_DOUBLE)

        # Display table with mean ± std
        display_df = agg_df.copy()
        display_df["val_loss"] = display_df.apply(
            lambda r: f"{r['mean_val_loss']:.4f} ± {r['std_val_loss']:.4f}", axis=1
        )
        display_df["val_ppl"] = display_df.apply(
            lambda r: f"{r['mean_val_ppl']:.2f} ± {r['std_val_ppl']:.2f}", axis=1
        )
        show_cols = ["config", "n_params", "val_loss", "val_ppl", "mean_runtime"]
        show_cols += [c for c in extra_cols if c in display_df.columns]
        display(display_df[[c for c in show_cols if c in display_df.columns]])

        # Bar chart with error bars
        fig, ax = plt.subplots(figsize=(8, 4))
        colors = ["#2ecc71" if i == 0 else "#3498db" for i in range(len(agg_df))]
        ax.bar(
            agg_df["config"],
            agg_df["mean_val_ppl"],
            yerr=agg_df["std_val_ppl"],
            color=colors,
            capsize=5,
        )
        ax.set_ylabel("Validation Perplexity (mean ± std, lower is better)")
        ax.set_title(title)
        plt.xticks(rotation=35, ha="right")
        plt.tight_layout()

        plot_path = _save_phase_plot(fig, title=title, phase_hint=phase_hint)
        print(f"  Saved phase plot: {plot_path.resolve()}")

        plt.show()

        winner = agg_df.iloc[0]
        print(
            f"  Winner: {winner['config']}  "
            f"(val_ppl={winner['mean_val_ppl']:.2f} ± {winner['std_val_ppl']:.2f})"
        )
        return agg_df
    else:
        # Single-seed path (backwards compatible)
        df = df.sort_values("best_val_loss")
        base_cols = ["run_name", "n_params", "best_val_loss", "best_val_ppl", "runtime_sec"]
        extra_cols = sorted(
            c for c in df.columns
            if c.startswith("model.") or c.startswith("training.")
        )
        show_cols = [c for c in base_cols + extra_cols if c in df.columns]

        print(f"\n{_SEP_DOUBLE}")
        print(f"  {title}")
        print(_SEP_DOUBLE)
        display(df[show_cols])

        fig, ax = plt.subplots(figsize=(8, 4))
        colors = ["#2ecc71" if i == 0 else "#3498db" for i in range(len(df))]
        ax.bar(df["run_name"], df["best_val_ppl"], color=colors)
        ax.set_ylabel("Validation Perplexity (lower is better)")
        ax.set_title(title)
        plt.xticks(rotation=35, ha="right")
        plt.tight_layout()

        plot_path = _save_phase_plot(fig, title=title, phase_hint=phase_hint)
        print(f"  Saved phase plot: {plot_path.resolve()}")

        plt.show()

        winner = df.iloc[0]
        print(f"  Winner: {winner['run_name']}  (val_ppl={winner['best_val_ppl']:.2f})")
        return df


---
## Phase 0: Multi-Scale Architecture Screening

**Goal:** Cheap micro-scale experiments (~7M params, 1000 steps each) that screen out bad architectural choices before investing in full-scale experiments.

**Time estimate:** ~25 min/run on MPS (M4 Pro), ~3 min/run on H100. Full phase: ~3.3h MPS, ~20 min H100.

**Method:** Single seed first. Only add multi-seed for ambiguous results.

| Run | d | L | heads | d_ff | act | pos_enc | qk_norm | dropout | Purpose |
|-----|---|---|-------|------|-----|---------|---------|---------|---------|
| p0_baseline | 128 | 4 | 4 | 341 | swiglu | rope | false | 0.0 | Baseline |
| p0_gelu | 128 | 4 | 4 | 512 | gelu | rope | false | 0.0 | SwiGLU vs GELU (iso-param) |
| p0_learned_pe | 128 | 4 | 4 | 341 | swiglu | learned | false | 0.0 | RoPE vs learned |
| p0_dropout01 | 128 | 4 | 4 | 341 | swiglu | rope | false | 0.1 | Dropout effect |
| p0_deeper | 128 | 6 | 4 | 227 | swiglu | rope | false | 0.0 | Deeper (6L, iso-param) |
| p0_wider | 160 | 3 | 4 | 341 | swiglu | rope | false | 0.0 | Wider (3L, d=160) |
| p0_qknorm | 128 | 4 | 4 | 341 | swiglu | rope | true | 0.0 | QK-norm effect |
| p0_lr_low | 128 | 4 | 4 | 341 | swiglu | rope | false | 0.0 | LR=3e-4 (vs 1e-3) |

> **Iso-param note:** GELU uses d_ff=512 (vs SwiGLU d_ff=341) to match parameter count, since SwiGLU has ~50% more FFN parameters per d_ff unit. The deeper variant (6L) uses d_ff=227 to stay at ~7M params.

In [ ]:
# ── Phase 0: Micro-Scale Architecture Screening (8 configs × 1 seed) ──

# Phase 0 uses smaller batch + grad_accum to fit on any device
P0_BATCH_SIZE = 16
P0_GRAD_ACCUM = 2   # 32K tokens/step — fits on any device
P0_SEED = 42         # single seed for fast screening
P0_LR = 1e-3         # small models need higher LR
P0_WARMUP = 50

# Shared micro-scale base overrides
_p0_base = {
    "model.d_model": 128,
    "model.n_layers": 4,
    "model.n_heads": 4,
    "model.d_ff": 341,
    "model.dropout": 0.0,
    "model.activation": "swiglu",
    "model.positional_encoding": "rope",
    "model.qk_norm": False,
    "training.learning_rate": P0_LR,
    "training.warmup_steps": P0_WARMUP,
    "training.batch_size": P0_BATCH_SIZE,
    "training.grad_accum_steps": P0_GRAD_ACCUM,
}

phase_0_experiments = [
    {
        "name": "p0_baseline",
        "overrides": {**_p0_base},
    },
    {
        "name": "p0_gelu",
        "overrides": {**_p0_base, "model.activation": "gelu", "model.d_ff": 512},
    },
    {
        "name": "p0_learned_pe",
        "overrides": {**_p0_base, "model.positional_encoding": "learned"},
    },
    {
        "name": "p0_dropout01",
        "overrides": {**_p0_base, "model.dropout": 0.1},
    },
    {
        "name": "p0_deeper",
        "overrides": {**_p0_base, "model.n_layers": 6, "model.d_ff": 227},
    },
    {
        "name": "p0_wider",
        "overrides": {**_p0_base, "model.d_model": 160, "model.n_layers": 3},
    },
    {
        "name": "p0_qknorm",
        "overrides": {**_p0_base, "model.qk_norm": True},
    },
    {
        "name": "p0_lr_low",
        "overrides": {**_p0_base, "training.learning_rate": 3e-4},
    },
]

phase_0_results = []
for exp in phase_0_experiments:
    result = run_ablation(
        run_name=f"{exp['name']}_s{P0_SEED}",
        overrides=exp["overrides"],
        max_steps=PHASE_0_STEPS,
        eval_interval=PHASE_0_EVAL,
        seed=P0_SEED,
    )
    phase_0_results.append(result)

In [ ]:
p0_df = compare_phase(phase_0_results, "Phase 0: Micro-Scale Architecture Screening")

print("\nKey takeaways to check:")
print("  - Does SwiGLU beat GELU at iso-param? (p0_baseline vs p0_gelu)")
print("  - Does RoPE beat learned PE? (p0_baseline vs p0_learned_pe)")
print("  - Does dropout help or hurt at this scale? (p0_baseline vs p0_dropout01)")
print("  - Is deeper better at iso-param? (p0_baseline vs p0_deeper)")
print("  - Does QK-norm help? (p0_baseline vs p0_qknorm)")
print("  - Is 1e-3 better than 3e-4 for micro models? (p0_baseline vs p0_lr_low)")

---
## Phase 0.5: Depth vs Width at Medium Scale

**Goal:** After Phase 0 eliminates bad architectural choices, test depth/width trade-offs at two scales (~12M and ~19M params) to guide full-scale config selection.

**Time estimate:** ~42 min/run on MPS (M4 Pro), ~4 min/run on H100. Full phase: ~4.2h MPS, ~25 min H100.

All configs use the Phase 0 winners (expected: SwiGLU + RoPE + no dropout at this scale).

| Run | d | L | heads | d_ff | ~Params |
|-----|---|---|-------|------|---------|
| p05_d192_L6 | 192 | 6 | 6 | 512 | ~12.3M |
| p05_d192_L8 | 192 | 8 | 6 | 384 | ~12.3M |
| p05_d192_L10 | 192 | 10 | 6 | 307 | ~12.3M |
| p05_d256_L6 | 256 | 6 | 8 | 910 | ~19.2M |
| p05_d256_L8 | 256 | 8 | 8 | 683 | ~19.2M |
| p05_d256_L10 | 256 | 10 | 8 | 546 | ~19.2M |

> **Iso-param pairs:** Within each scale (12M and 19M), d_ff is adjusted so total params stay roughly constant as depth increases. This isolates the depth vs width trade-off.

In [ ]:
# ── Phase 0.5: Depth vs Width at Medium Scale (6 configs × 1 seed) ──

P05_SEED = 42
P05_LR = 6e-4   # medium models: lower LR than micro
P05_WARMUP = 100

phase_05_experiments = [
    # ~12.3M param tier (d=192)
    {
        "name": "p05_d192_L6",
        "overrides": {
            "model.d_model": 192, "model.n_layers": 6, "model.n_heads": 6,
            "model.d_ff": 512, "model.dropout": 0.0,
            "training.learning_rate": P05_LR, "training.warmup_steps": P05_WARMUP,
            "training.batch_size": P0_BATCH_SIZE, "training.grad_accum_steps": P0_GRAD_ACCUM,
        },
    },
    {
        "name": "p05_d192_L8",
        "overrides": {
            "model.d_model": 192, "model.n_layers": 8, "model.n_heads": 6,
            "model.d_ff": 384, "model.dropout": 0.0,
            "training.learning_rate": P05_LR, "training.warmup_steps": P05_WARMUP,
            "training.batch_size": P0_BATCH_SIZE, "training.grad_accum_steps": P0_GRAD_ACCUM,
        },
    },
    {
        "name": "p05_d192_L10",
        "overrides": {
            "model.d_model": 192, "model.n_layers": 10, "model.n_heads": 6,
            "model.d_ff": 307, "model.dropout": 0.0,
            "training.learning_rate": P05_LR, "training.warmup_steps": P05_WARMUP,
            "training.batch_size": P0_BATCH_SIZE, "training.grad_accum_steps": P0_GRAD_ACCUM,
        },
    },
    # ~19.2M param tier (d=256)
    {
        "name": "p05_d256_L6",
        "overrides": {
            "model.d_model": 256, "model.n_layers": 6, "model.n_heads": 8,
            "model.d_ff": 910, "model.dropout": 0.0,
            "training.learning_rate": P05_LR, "training.warmup_steps": P05_WARMUP,
            "training.batch_size": P0_BATCH_SIZE, "training.grad_accum_steps": P0_GRAD_ACCUM,
        },
    },
    {
        "name": "p05_d256_L8",
        "overrides": {
            "model.d_model": 256, "model.n_layers": 8, "model.n_heads": 8,
            "model.d_ff": 683, "model.dropout": 0.0,
            "training.learning_rate": P05_LR, "training.warmup_steps": P05_WARMUP,
            "training.batch_size": P0_BATCH_SIZE, "training.grad_accum_steps": P0_GRAD_ACCUM,
        },
    },
    {
        "name": "p05_d256_L10",
        "overrides": {
            "model.d_model": 256, "model.n_layers": 10, "model.n_heads": 8,
            "model.d_ff": 546, "model.dropout": 0.0,
            "training.learning_rate": P05_LR, "training.warmup_steps": P05_WARMUP,
            "training.batch_size": P0_BATCH_SIZE, "training.grad_accum_steps": P0_GRAD_ACCUM,
        },
    },
]

phase_05_results = []
for exp in phase_05_experiments:
    result = run_ablation(
        run_name=f"{exp['name']}_s{P05_SEED}",
        overrides=exp["overrides"],
        max_steps=PHASE_05_STEPS,
        eval_interval=PHASE_05_EVAL,
        seed=P05_SEED,
    )
    phase_05_results.append(result)

In [ ]:
p05_df = compare_phase(phase_05_results, "Phase 0.5: Depth vs Width at Medium Scale")

print("\nKey takeaways to check:")
print("  - Within 12M tier: does depth (L=10) or width (L=6) win?")
print("  - Within 19M tier: same question at larger scale")
print("  - Does the optimal depth/width ratio change with scale?")
print("  - These results should guide Phase 1 config selection")

---
## Phase 1: Architecture Shape

**Question:** What depth / width / FFN ratio works best at ~37-40M params?

**Fixed:** SwiGLU, dropout=0.0, LR=6e-4, cosine schedule, AdamW, weight_tying

| Variant | d_model | Layers | Heads | d_ff | ~Params |
|---------|--------:|-------:|------:|-----:|--------:|
| A: Balanced | 384 | 10 | 12 | 1024 | 37.0M |
| B: Deeper, tighter FFN | 384 | 12 | 12 | 896 | 38.8M |
| C: Max budget | 384 | 12 | 12 | 1024 | 40.5M |
| D: Deep-narrow | 320 | 16 | 8 | 853 | 35.8M |

Each variant is trained with **multiple seeds** to report mean ± std and ensure
ranking differences are not due to initialization luck.

In [ ]:
# ── Phase 1: Architecture Shape (device-tiered, eval OOM-safe) ───────

if device.type == "cuda":
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if vram_gb >= 70:
        # A100 80GB class
        P1_TRAIN_BATCH, P1_GRAD_ACCUM, P1_EVAL_BATCH = 64, 4, 32
    elif vram_gb >= 35:
        P1_TRAIN_BATCH, P1_GRAD_ACCUM, P1_EVAL_BATCH = 32, 4, 16
    else:
        P1_TRAIN_BATCH, P1_GRAD_ACCUM, P1_EVAL_BATCH = 16, 4, 8
elif device.type == "mps":
    P1_TRAIN_BATCH, P1_GRAD_ACCUM, P1_EVAL_BATCH = 16, 4, 8
else:
    P1_TRAIN_BATCH, P1_GRAD_ACCUM, P1_EVAL_BATCH = 8, 8, 4

print(
    f"Phase 1 batch config: train={P1_TRAIN_BATCH}, grad_accum={P1_GRAD_ACCUM}, "
    f"eval={P1_EVAL_BATCH}"
)

p1_common = {
    "model.dropout": 0.0,
    "training.learning_rate": 6e-4,
    "training.batch_size": P1_TRAIN_BATCH,
    "training.grad_accum_steps": P1_GRAD_ACCUM,
    "training.eval_batch_size": P1_EVAL_BATCH,
}

phase_1_experiments = [
    {
        "name": "p1_shape_A_d384_L10_ff1024",
        "overrides": {
            **p1_common,
            "model.d_model": 384,
            "model.n_layers": 10,
            "model.n_heads": 12,
            "model.d_ff": 1024,
        },
    },
    {
        "name": "p1_shape_B_d384_L12_ff896",
        "overrides": {
            **p1_common,
            "model.d_model": 384,
            "model.n_layers": 12,
            "model.n_heads": 12,
            "model.d_ff": 896,
        },
    },
    {
        "name": "p1_shape_C_d384_L12_ff1024",
        "overrides": {
            **p1_common,
            "model.d_model": 384,
            "model.n_layers": 12,
            "model.n_heads": 12,
            "model.d_ff": 1024,
        },
    },
    {
        "name": "p1_shape_D_d320_L16_ff853",
        "overrides": {
            **p1_common,
            "model.d_model": 320,
            "model.n_layers": 16,
            "model.n_heads": 8,
            "model.d_ff": 853,
        },
    },
]

phase_1_results = []
for exp in phase_1_experiments:
    seed_results = run_ablation_multi_seed(
        base_name=exp["name"],
        overrides=exp["overrides"],
        max_steps=PHASE_1_STEPS,
        eval_interval=PHASE_1_EVAL,
    )
    phase_1_results.extend(seed_results)


In [ ]:
p1_df = compare_phase(phase_1_results, "Phase 1: Architecture Shape")

# Extract winner for Phase 2
p1_winner = p1_df.iloc[0]

# Handle both multi-seed (aggregated) and single-seed DataFrames
if "config" in p1_winner.index:
    # Multi-seed: columns are config, mean_val_loss, etc. + model.* overrides
    print(f"\nUsing Phase 1 winner for Phase 2:")
    print(f"  Config: {p1_winner['config']}")
    print(f"  mean_val_ppl={p1_winner['mean_val_ppl']:.2f} ± {p1_winner['std_val_ppl']:.2f}")
else:
    print(f"\nUsing Phase 1 winner for Phase 2:")
    print(f"  d_model={p1_winner.get('model.d_model')}, "
          f"n_layers={p1_winner.get('model.n_layers')}, "
          f"n_heads={p1_winner.get('model.n_heads')}, "
          f"d_ff={p1_winner.get('model.d_ff')}")

---
## Phase 1.5A / 1.5B: Shape C Local Refinement

**Goal:** Use Phase 1 Shape C (`d384, L12, d_ff=1024`) as template and run local architecture variants.

**Structure:**
- **Phase 1.5A**: archived legacy setup (kept disabled and commented).
- **Phase 1.5B**: active one-factor sweeps (L-only, ff-only, d-only).

**Important:** The original Shape C baseline is intentionally **not** rerun here.

| Variant | d_model | Layers | Heads | d_ff | Purpose |
|---------|--------:|-------:|------:|-----:|---------|
| C-wide-shallow | 384 | 10 | 12 | 1229 | More FFN capacity, less depth |
| C-deep-balanced | 384 | 14 | 12 | 878 | More depth, reduced FFN |
| C-very-deep-lean | 384 | 16 | 12 | 768 | Strong depth bias |


In [ ]:
# ── Phase 1.5A: Shape C Local Refinement (DISABLED ARCHIVE) ──
# This block is intentionally disabled to prevent accidental re-execution.
#
# Keep Phase 1 training settings unless overridden.
# p15_common = dict(globals().get("p1_common", {
#     "model.dropout": 0.0,
#     "training.learning_rate": 6e-4,
#     "training.batch_size": 8,
#     "training.grad_accum_steps": 8,
#     "training.eval_batch_size": 8,
# }))
#
# p15_base = {
#     **p15_common,
#     "model.d_model": 384,
#     "model.n_heads": 12,
# }
#
# # Original Shape C (L12, d_ff=1024) was intentionally excluded in this archived block.
# phase_1_5_experiments = [
#     {
#         "name": "p15_shape_C_wide_shallow_d384_L10_ff1229",
#         "overrides": {
#             **p15_base,
#             "model.n_layers": 10,
#             "model.d_ff": 1229,
#         },
#     },
#     {
#         "name": "p15_shape_C_deep_balanced_d384_L14_ff878",
#         "overrides": {
#             **p15_base,
#             "model.n_layers": 14,
#             "model.d_ff": 878,
#         },
#     },
#     {
#         "name": "p15_shape_C_very_deep_lean_d384_L16_ff768",
#         "overrides": {
#             **p15_base,
#             "model.n_layers": 16,
#             "model.d_ff": 768,
#         },
#     },
# ]
#
# phase_1_5_results = []
# for exp in phase_1_5_experiments:
#     seed_results = run_ablation_multi_seed(
#         base_name=exp["name"],
#         overrides=exp["overrides"],
#         max_steps=PHASE_15_STEPS,
#         eval_interval=PHASE_15_EVAL,
#     )
#     phase_1_5_results.extend(seed_results)


In [ ]:
# ── Phase 1.5B: Shape C L-only, ff-only, and d-only sweeps ──
# One-factor sweeps around Shape C template.
# - L sweep: d=384, ff=1024, L in [10, 12, 14]
# - ff sweep: d=384, L=12, ff in [896, 1024, 1152]
# - d sweep: L=12, ff=1024, d in [320, 384, 448]
# Duplicate architecture combinations are deduplicated automatically,
# preserving the first occurrence (L-sweep entries are kept first).


# NOTE ON COMPARABILITY:
# Phase 1.5B is run under a different training regime than the earlier Phase 1 runs
# (higher effective tokens per step due to larger batch settings on A100).
# Therefore, repeated architecture points (e.g., L=12, ff=896/1024, d=384) are treated
# as re-evaluations under this new regime, not strict duplicates.

# Interpret Phase 1 vs Phase 1.5A/B differences with caution when ranking close variants.



p15_common = dict(globals().get("p1_common", {
    "model.dropout": 0.0,
    "training.learning_rate": 6e-4,
    "training.batch_size": BATCH_SIZE,
    "training.grad_accum_steps": GRAD_ACCUM,
    "training.eval_batch_size": EVAL_BATCH_SIZE,
}))

phase_1_5_experiments = []

# L-only sweep (run first)
for n_layers in [10, 12, 14]:
    phase_1_5_experiments.append({
        "name": f"p15_shape_C_L_only_d384_L{n_layers}_ff1024",
        "overrides": {
            **p15_common,
            "model.d_model": 384,
            "model.n_heads": 12,
            "model.n_layers": n_layers,
            "model.d_ff": 1024,
        },
    })

# ff-only sweep
for ff in [896, 1024, 1152]:
    phase_1_5_experiments.append({
        "name": f"p15_shape_C_ff_only_d384_L12_ff{ff}",
        "overrides": {
            **p15_common,
            "model.d_model": 384,
            "model.n_heads": 12,
            "model.n_layers": 12,
            "model.d_ff": ff,
        },
    })

# d-only sweep
for d_model in [320, 384, 448]:
    n_heads = d_model // 32
    phase_1_5_experiments.append({
        "name": f"p15_shape_C_d_only_d{d_model}_L12_ff1024",
        "overrides": {
            **p15_common,
            "model.d_model": d_model,
            "model.n_heads": n_heads,
            "model.n_layers": 12,
            "model.d_ff": 1024,
        },
    })

# Deduplicate identical architecture settings (e.g., baseline overlap).
unique = {}
for exp in phase_1_5_experiments:
    o = exp["overrides"]
    key = (o["model.d_model"], o["model.n_layers"], o["model.d_ff"])
    if key not in unique:
        unique[key] = exp
phase_1_5_experiments = list(unique.values())

print("Phase 1.5B experiment set:")
for exp in phase_1_5_experiments:
    o = exp["overrides"]
    print(
        f"  - {exp['name']}: d={o['model.d_model']}, "
        f"L={o['model.n_layers']}, ff={o['model.d_ff']}, heads={o['model.n_heads']}"
    )

phase_1_5_results = []
for exp in phase_1_5_experiments:
    seed_results = run_ablation_multi_seed(
        base_name=exp["name"],
        overrides=exp["overrides"],
        max_steps=PHASE_15_STEPS,
        eval_interval=PHASE_15_EVAL,
    )
    phase_1_5_results.extend(seed_results)


In [ ]:
p15_df = compare_phase(phase_1_5_results, "Phase 1.5B: Shape C Local Refinement")

# Extract winner for Phase 2.
p1_5_winner = p15_df.iloc[0]

print("\nUsing Phase 1.5B winner for Phase 2:")
print(f"  Config: {p1_5_winner.get('config', 'n/a')}")
if "mean_val_ppl" in p1_5_winner.index:
    print(f"  mean_val_ppl={p1_5_winner['mean_val_ppl']:.2f} ± {p1_5_winner['std_val_ppl']:.2f}")


---
## Phase 2: Learning Rate Sweep

Takes the winning architecture from Phase 1.5B (or Phase 1 if Phase 1.5B was skipped).

**Sweep:** 3e-4, 6e-4, 1e-3

> **Note:** The cell below automatically uses the Phase 1.5B winner when available.
> If you want to override manually, edit `P2_ARCH` below.



In [ ]:
# ── Phase 2: Learning Rate Sweep (3 configs × N seeds) ─────────────

if "p1_winner" in globals():
    arch_winner = p1_winner
    arch_source = "Phase 1"
elif IN_COLAB:
    # Hardcoded fallback: best Phase 1 architecture (Shape C)
    arch_winner = {
        "model.d_model": 384,
        "model.n_layers": 12,
        "model.n_heads": 12,
        "model.d_ff": 1024,
    }
    arch_source = "Phase 1 hardcoded fallback (Shape C, Colab)"
else:
    raise ValueError("No architecture winner found. Run Phase 1 or use Colab fallback.")

P2_ARCH = {
    "model.d_model": int(arch_winner.get("model.d_model", 384)),
    "model.n_layers": int(arch_winner.get("model.n_layers", 10)),
    "model.n_heads": int(arch_winner.get("model.n_heads", 12)),
    "model.d_ff": int(arch_winner.get("model.d_ff", 1024)),
    "model.dropout": 0.0,
}

print(f"Using {arch_source} winner for Phase 2 architecture:")
print(
    f"  d_model={P2_ARCH['model.d_model']}, "
    f"n_layers={P2_ARCH['model.n_layers']}, "
    f"n_heads={P2_ARCH['model.n_heads']}, "
    f"d_ff={P2_ARCH['model.d_ff']}"
)

LR_CANDIDATES = [3e-4, 6e-4, 1e-3]

phase_2_results = []
for lr in LR_CANDIDATES:
    overrides = {**P2_ARCH, "training.learning_rate": lr}
    seed_results = run_ablation_multi_seed(
        base_name=f"p2_lr_{lr}",
        overrides=overrides,
        max_steps=PHASE_2_STEPS,
        eval_interval=PHASE_2_EVAL,
    )
    phase_2_results.extend(seed_results)


In [ ]:
p2_df = compare_phase(phase_2_results, "Phase 2: Learning Rate Sweep")

# Extract winner for Phase 3
p2_winner = p2_df.iloc[0]

if "config" in p2_winner.index:
    best_lr = p2_winner.get("training.learning_rate", 6e-4)
    print(f"\nUsing best LR for Phase 3: {best_lr}")
    print(f"  mean_val_ppl={p2_winner['mean_val_ppl']:.2f} ± {p2_winner['std_val_ppl']:.2f}")
else:
    best_lr = p2_winner.get("training.learning_rate", 6e-4)
    print(f"\nUsing best LR for Phase 3: {best_lr}")

---
## Phase 3: Coupled Ablations

Uses winning architecture + LR from Phases 1 & 2.

| Experiment | What changes | Expected |
|-----------|-------------|----------|
| Dropout 0.1 | `model.dropout=0.1` | May help if overfitting, hurt otherwise |
| QK-Norm | `model.qk_norm=true` | Stabilises deep models (L>=12) |
| GELU (vs SwiGLU) | `model.activation=gelu` | SwiGLU should win (sanity check) |

> The dropout=0.0 baseline was already run in Phase 1/2, so we only run 0.1.

In [ ]:
# ── Phase 3: Coupled Ablations (3 configs × N seeds) ───────────────

# Base config from Phase 1 + Phase 2 winners
P3_BASE = {
    **P2_ARCH,
    "training.learning_rate": best_lr,
}

phase_3_experiments = [
    {
        "name": "p3_dropout_0.1",
        "overrides": {**P3_BASE, "model.dropout": 0.1},
    },
    {
        "name": "p3_qknorm",
        "overrides": {**P3_BASE, "model.qk_norm": True},
    },
    {
        "name": "p3_gelu",
        "overrides": {**P3_BASE, "model.activation": "gelu"},
    },
]

phase_3_results = []
for exp in phase_3_experiments:
    seed_results = run_ablation_multi_seed(
        base_name=exp["name"],
        overrides=exp["overrides"],
        max_steps=PHASE_3_STEPS,
        eval_interval=PHASE_3_EVAL,
    )
    phase_3_results.extend(seed_results)

In [ ]:
p3_df = compare_phase(phase_3_results, "Phase 3: Coupled Ablations")

---
## Overall Summary

In [ ]:
# ── Final summary: visualize results + save artifacts (+ Drive backup in Colab) ──
from pathlib import Path
from datetime import datetime
import json
import shutil
import pickle
import random
import pandas as pd
import matplotlib.pyplot as plt

if "all_results" not in globals() or not all_results:
    raise ValueError("all_results is empty. Run experiment phases first.")

RUNS_DIR = Path(globals().get("RUNS_DIR", Path("runs")))
RUNS_DIR.mkdir(parents=True, exist_ok=True)
out_dir = RUNS_DIR / "visualizations" / "all_results"
out_dir.mkdir(parents=True, exist_ok=True)

summary_df = pd.DataFrame(all_results).copy()
summary_df["config_key"] = summary_df["run_name"].str.replace(r"_s\d+$", "", regex=True)

agg_rows = []
for config_key, group in summary_df.groupby("config_key"):
    phase = config_key.split("_")[0]
    agg_rows.append({
        "config": config_key,
        "phase": phase,
        "n_seeds": len(group),
        "n_params": group["n_params"].iloc[0],
        "mean_val_loss": group["best_val_loss"].mean(),
        "std_val_loss": group["best_val_loss"].std(),
        "mean_val_ppl": group["best_val_ppl"].mean(),
        "std_val_ppl": group["best_val_ppl"].std(),
        "total_runtime_sec": group["runtime_sec"].sum(),
    })

agg_df = pd.DataFrame(agg_rows).sort_values("mean_val_loss")

print(f"All ablation runs ({len(summary_df)} total):")
display(
    summary_df[
        ["run_name", "seed", "n_params", "best_val_loss", "best_val_ppl", "max_steps", "runtime_sec"]
    ].sort_values("best_val_loss")
)

print("\nAggregated results (mean ± std across seeds):")
agg_display = agg_df.copy()
agg_display["val_loss"] = agg_display.apply(
    lambda r: f"{r['mean_val_loss']:.4f} ± {0.0 if pd.isna(r['std_val_loss']) else r['std_val_loss']:.4f}",
    axis=1,
)
agg_display["val_ppl"] = agg_display.apply(
    lambda r: f"{r['mean_val_ppl']:.2f} ± {0.0 if pd.isna(r['std_val_ppl']) else r['std_val_ppl']:.2f}",
    axis=1,
)
display(agg_display[["config", "phase", "n_params", "val_loss", "val_ppl", "n_seeds", "total_runtime_sec"]])

# Save core tables
all_json = RUNS_DIR / "ablation_summary.json"
all_csv = out_dir / "all_runs.csv"
agg_csv = out_dir / "all_runs_aggregated.csv"
agg_md = out_dir / "all_runs_aggregated.md"

all_json.write_text(json.dumps(all_results, indent=2, default=str), encoding="utf-8")
summary_df.to_csv(all_csv, index=False)
agg_df.to_csv(agg_csv, index=False)
try:
    agg_df.to_markdown(agg_md, index=False)
except Exception:
    pass

# Save additional context artifacts
phase_tables = {
    "p1_df": globals().get("p1_df"),
    "p15_df": globals().get("p15_df"),
    "p2_df": globals().get("p2_df"),
    "p3_df": globals().get("p3_df"),
}
for name, table in phase_tables.items():
    if table is not None:
        try:
            table.to_csv(out_dir / f"{name}.csv", index=False)
        except Exception:
            pass

winners = {}
for key in ["p1_winner", "p1_5_winner", "p2_winner"]:
    if key in globals():
        val = globals()[key]
        winners[key] = val.to_dict() if hasattr(val, "to_dict") else str(val)
(out_dir / "winners.json").write_text(json.dumps(winners, indent=2, default=str), encoding="utf-8")

run_context = {
    "timestamp": datetime.now().isoformat(),
    "seeds": globals().get("SEEDS"),
    "config_path": globals().get("CONFIG_PATH"),
    "device": str(globals().get("device", globals().get("DEVICE", "unknown"))),
    "phase_steps": {
        "PHASE_0_STEPS": globals().get("PHASE_0_STEPS"),
        "PHASE_05_STEPS": globals().get("PHASE_05_STEPS"),
        "PHASE_1_STEPS": globals().get("PHASE_1_STEPS"),
        "PHASE_15_STEPS": globals().get("PHASE_15_STEPS"),
        "PHASE_2_STEPS": globals().get("PHASE_2_STEPS"),
        "PHASE_3_STEPS": globals().get("PHASE_3_STEPS"),
    },
}
(out_dir / "run_context.json").write_text(json.dumps(run_context, indent=2, default=str), encoding="utf-8")

# Plot overview
phase_colors = {"p0": "#f39c12", "p05": "#9b59b6", "p1": "#3498db", "p15": "#1abc9c", "p2": "#e74c3c", "p3": "#2ecc71"}
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
colors = [phase_colors.get(row["phase"], "#95a5a6") for _, row in agg_df.iterrows()]
ax.barh(
    agg_df["config"],
    agg_df["mean_val_ppl"],
    xerr=agg_df["std_val_ppl"].fillna(0.0),
    color=colors,
    capsize=4,
)
ax.set_xlabel("Validation Perplexity (mean ± std, lower is better)")
ax.set_title("All Ablations (aggregated across seeds)")
ax.invert_yaxis()

ax = axes[1]
ax.errorbar(
    agg_df["n_params"] / 1e6,
    agg_df["mean_val_ppl"],
    yerr=agg_df["std_val_ppl"].fillna(0.0),
    fmt="o",
    markersize=8,
    capsize=4,
    color="#34495e",
    ecolor="#bdc3c7",
)
for _, row in agg_df.iterrows():
    label = row["config"].split("_", 2)[-1] if "_" in row["config"] else row["config"]
    ax.annotate(label, (row["n_params"] / 1e6, row["mean_val_ppl"]), fontsize=7, ha="left", va="bottom")

ax.set_xlabel("Parameters (M)")
ax.set_ylabel("Validation Perplexity (mean ± std)")
ax.set_title("Params vs Performance")

plt.tight_layout()
plot_path = out_dir / "all_ablations_overview.png"
fig.savefig(plot_path, dpi=180, bbox_inches="tight")
plt.show()

best = agg_df.iloc[0]
print(
    f"\nBest overall: {best['config']} | "
    f"val_ppl={best['mean_val_ppl']:.2f} ± {0.0 if pd.isna(best['std_val_ppl']) else best['std_val_ppl']:.2f} | "
    f"params={best['n_params']/1e6:.2f}M"
)

# Build artifact list for copy/sync
artifact_files = [all_json, all_csv, agg_csv, plot_path, out_dir / "winners.json", out_dir / "run_context.json"]
if agg_md.exists():
    artifact_files.append(agg_md)
for optional_name in ["p1_df.csv", "p15_df.csv", "p2_df.csv", "p3_df.csv"]:
    opt = out_dir / optional_name
    if opt.exists():
        artifact_files.append(opt)

# Add all saved phase plot files (compare_phase saves these automatically)
for plot_file in globals().get("PHASE_PLOT_FILES", []):
    p = Path(plot_file)
    if p.exists():
        artifact_files.append(p)

# Add any PNG found under runs/visualizations as a safety net.
for png in sorted((RUNS_DIR / "visualizations").rglob("*.png")):
    artifact_files.append(png)

# Deduplicate artifact list while preserving order.
seen = set()
unique_artifacts = []
for f in artifact_files:
    s = str(f)
    if s not in seen and Path(f).exists():
        seen.add(s)
        unique_artifacts.append(Path(f))
artifact_files = unique_artifacts

if IN_COLAB:
    drive_root = Path("/content/drive/MyDrive/tiny_llm")
    
    print(f"Detected Colab environment. Preparing to copy/sync artifacts to Drive.")

    # 1) Copy selected summary artifacts and plots for quick access.
    drive_viz_dir = drive_root / "visualisations" / "all_results"
    drive_viz_dir.mkdir(parents=True, exist_ok=True)
    for src in artifact_files:
        shutil.copy2(src, drive_viz_dir / src.name)

    # 2) Sync full run-group directory (includes checkpoints, metrics, summaries, and visuals).
    drive_runs_root = drive_root / "runs"
    drive_runs_root.mkdir(parents=True, exist_ok=True)
    drive_runs_dir = drive_runs_root / RUNS_DIR.name
    shutil.copytree(RUNS_DIR, drive_runs_dir, dirs_exist_ok=True)

    # 3) Create timestamped backup snapshot with full runs + memory context.
    backup_dir = drive_root / "backups" / f"ablation_backup_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    (backup_dir / "memory").mkdir(parents=True, exist_ok=True)

    backup_runs_dir = backup_dir / "runs" / RUNS_DIR.name
    backup_runs_dir.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(RUNS_DIR, backup_runs_dir, dirs_exist_ok=True)

    important_globals = [
        "all_results", "phase_1_results", "phase_1_5_results", "phase_2_results", "phase_3_results",
        "p1_winner", "p1_5_winner", "p2_winner", "SEEDS", "CONFIG_PATH", "DEVICE", "PHASE_PLOT_FILES",
    ]
    saved_globals = {}
    for key in important_globals:
        if key not in globals():
            continue
        value = globals()[key]
        pkl_path = backup_dir / "memory" / f"{key}.pkl"
        json_path = backup_dir / "memory" / f"{key}.json"
        try:
            with open(pkl_path, "wb") as f:
                pickle.dump(value, f)
            with open(json_path, "w", encoding="utf-8") as f:
                if hasattr(value, "to_dict"):
                    json.dump(value.to_dict(), f, indent=2, default=str)
                else:
                    json.dump(value, f, indent=2, default=str)
            saved_globals[key] = "pkl+json"
        except Exception as exc:
            saved_globals[key] = f"failed ({exc})"

    # Save RNG state for reproducibility.
    rng_state = {"python_random_state": random.getstate()}
    try:
        import numpy as np
        rng_state["numpy_random_state"] = np.random.get_state()
    except Exception:
        pass
    try:
        import torch
        rng_state["torch_rng_state_cpu"] = torch.get_rng_state().tolist()
        if torch.cuda.is_available():
            rng_state["torch_rng_state_cuda"] = [x.tolist() for x in torch.cuda.get_rng_state_all()]
    except Exception:
        pass
    with open(backup_dir / "memory" / "rng_state.pkl", "wb") as f:
        pickle.dump(rng_state, f)

    notebook_path = Path("notebooks/ablation_experiments.ipynb")
    if notebook_path.exists():
        (backup_dir / "notebooks").mkdir(parents=True, exist_ok=True)
        shutil.copy2(notebook_path, backup_dir / "notebooks" / notebook_path.name)

    metrics_count = len(list((backup_dir / "runs").rglob("*_metric.json")))
    checkpoint_count = len(list((backup_dir / "runs").rglob("checkpoints/**/*")))

    manifest = {
        "timestamp": datetime.now().isoformat(),
        "drive_visualizations": str(drive_viz_dir),
        "drive_runs_sync": str(drive_runs_dir),
        "backup_dir": str(backup_dir),
        "saved_globals": saved_globals,
        "artifact_files": [str(p) for p in artifact_files],
        "runs_metrics_count": metrics_count,
        "checkpoint_files_count": checkpoint_count,
    }
    (backup_dir / "backup_manifest.json").write_text(json.dumps(manifest, indent=2, default=str), encoding="utf-8")

    print(f"\nSaved local: {out_dir.resolve()}")
    print(f"Copied summary artifacts to Drive: {drive_viz_dir.resolve()}")
    print(f"Synced full runs directory to Drive: {drive_runs_dir.resolve()}")
    print(f"Saved runtime backup snapshot to Drive: {backup_dir.resolve()}")
else:
    print(f"\nSaved local: {out_dir.resolve()}")

print(
    f"Best overall: {best['config']} "
    f"(val_ppl={best['mean_val_ppl']:.2f} ± {0.0 if pd.isna(best['std_val_ppl']) else best['std_val_ppl']:.2f})"
)


In [ ]:
if IN_COLAB:
    from google.colab import runtime
    print("Ending Colab session...")
    runtime.unassign()